# Phụ lục A: Hướng dẫn tái lập thực nghiệm

## A.1. Biên dịch lại báo cáo Jupyter Book

Toàn bộ bảng số liệu và biểu đồ trong báo cáo này được sinh tự động từ tệp tổng hợp `report_summary.json` cùng các đoạn mã gọi trực tiếp thư viện `parser` ngay trong quá trình biên dịch (`execute_notebooks: force` với cờ kiểm tra cảnh báo `-W`), không yêu cầu tải lại 22,72 GiB mã nguồn thô:

```bash
# Cách 1: Biên dịch trực tiếp từ kho lưu trữ Jupyter Book
python -m pip install -r requirements.txt
jupyter-book build . --all -W

# Cách 2: Sinh lại toàn bộ mã nguồn Book từ kho mã nguồn chính (nhánh main)
git clone https://github.com/NguyenQuocDat06-AI/arxiv-harvester.git
cd arxiv-harvester
python -m pip install -e ".[report]"
python tools/build_team_25000_book.py --workspace build/m2_workspace --output build/m2_book_staging --build
```

## A.2. Tái lập thu thập dữ liệu Milestone 1 (CLI và Google Colab CPU-only)

Trên máy cục bộ hoặc máy ảo Google Colab CPU-only (`colab_crawler.ipynb`, `kaggle_crawler.ipynb`, `tools/m1_colab_tri_resume.py`), cài đặt gói bằng `pip install -e .` để sử dụng các lệnh dòng lệnh:

```bash
# 1. Thu thập đa phiên bản và trích dẫn cho dải mã arXiv của một thành viên (hỗ trợ --resume)
arxiv-harvester               --member-id 24120282               --start-month 2024-12 --start-id 18848               --end-month 2025-01 --end-id 2641               --raw-dir data/raw               --output-dir data/shards/24120282               --export-course --resume

# 2. Kiểm toán độc lập mã nguồn TeX/Bib/BBL và tải bù qua HTTP Range
arxiv-audit-sources --publications data/shards --raw-dir data/raw --output data/audits/sources.parquet
arxiv-retry-sources --manifest data/audits/sources.retry.jsonl --output-dir data/repaired_sources --limit 20 --execute

# 3. Hợp nhất 5 phân vùng thành viên (25.000 bài) và xuất 2 bảng chuẩn M1 (papers, references)
python -m crawler.merge_shards --shards-dir data/shards --output-dir data/processed
```

## A.3. Tái lập phân tích cấu trúc và kiểm chuẩn Milestone 2 (Google Colab CPU-only)

1. **Thực thi theo từng phân vùng trên Colab CPU Standard**: mở `colab_parser_full.ipynb` để chạy `tools/m2_colab_runner.py` kèm `configs/parser_roots.json`, hoặc gọi trực tiếp qua CLI:

```bash
arxiv-parser               --publications data/processed/publications.parquet               --raw-dir data/raw               --output-dir data/parsed               --root-overrides configs/parser_roots.json               --resume

arxiv-validate-project               --course-dir data/course_export               --parsed-dir data/parsed               --publications data/processed/publications.parquet               --output data/parsed/project_validation.json               --require-full
```

2. **Hợp nhất và tổng hợp thông số 25.000 bài báo**: chạy `tools/m2_team_consolidator.py` để gộp kết quả 5 thành viên, kiểm chuẩn toàn vẹn 5 bảng bằng SQLite (`structural_valid = True`) và tổng hợp thông số 5 phiên Colab CPU qua `src/crawler/benchmark_sessions.py`.

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
import sys, importlib.metadata
pkgs = ["jupyter-book", "pandas", "pyarrow", "matplotlib", "nbformat", "arxiv-harvester"]
rows = [{"Thành phần": "Python", "Phiên bản khi biên dịch": sys.version.split()[0]}]
for p in pkgs:
    try:
        ver = importlib.metadata.version(p)
    except importlib.metadata.PackageNotFoundError:
        ver = "editable / PYTHONPATH"
    rows.append({"Thành phần": p, "Phiên bản khi biên dịch": ver})
display(style_table(pd.DataFrame(rows), caption="Bảng A.1. Phiên bản môi trường Python và các thư viện sử dụng khi biên dịch báo cáo."))

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`pyproject.toml`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/pyproject.toml).
- Mã nguồn triển khai: [`colab_crawler.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/colab_crawler.ipynb).
- Mã nguồn triển khai: [`colab_parser_full.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/colab_parser_full.ipynb).
- Mã nguồn triển khai: [`configs/parser_roots.json`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/configs/parser_roots.json).
- Mã nguồn triển khai: [`tools/m1_colab_tri_resume.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m1_colab_tri_resume.py).
- Mã nguồn triển khai: [`tools/m2_colab_runner.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).
- Mã nguồn triển khai: [`tools/extract_book_aggregates_colab.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/extract_book_aggregates_colab.py).